# NVIDIA NIM Quick Start

NIM Chat API는 OpenAI와 동일한 인터페이스입니다. `base_url`과 `api_key`만 변경하면 바로 사용할 수 있습니다.

## 0. 설치 & 설정

In [ ]:
!uv sync

In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("NVIDIA_BASE_URL", "http://localhost:8000/v1")
API_KEY = os.environ.get("NVIDIA_API_KEY", "no-key")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

CHAT_MODEL = "nvidia/nemotron-3-nano-30b-a3b"
# CHAT_MODEL = "nvidia/nemotron-3-super-120b-a12b"  # 고성능

print(f"Model: {CHAT_MODEL}")
print(f"Base URL: {BASE_URL}")

## 1. 기본 Chat Completion

In [ ]:
response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "당신은 친절한 AI 어시스턴트입니다."},
        {"role": "user",   "content": "NIM이 뭐야? 두 문장으로 설명해줘."},
    ],
    temperature=0.5,
    max_tokens=256,
)

print(response.choices[0].message.content)
print(f"\n[토큰] 입력={response.usage.prompt_tokens}, 출력={response.usage.completion_tokens}")

## 2. 스트리밍

In [ ]:
stream = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "당신은 친절한 AI 어시스턴트입니다."},
        {"role": "user",   "content": "Python의 장점을 3가지 알려줘."},
    ],
    temperature=0.5,
    max_tokens=256,
    stream=True,
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

## 3. 멀티턴 대화

In [ ]:
history = [{"role": "system", "content": "당신은 친절한 AI 어시스턴트입니다."}]

def chat(user_input: str) -> str:
    history.append({"role": "user", "content": user_input})
    resp = client.chat.completions.create(
        model=CHAT_MODEL, messages=history, temperature=0.5, max_tokens=256
    )
    reply = resp.choices[0].message.content
    history.append({"role": "assistant", "content": reply})
    return reply

print("Turn 1:", chat("내 이름은 준호야."))
print("\nTurn 2:", chat("내 이름이 뭐라고 했지?"))

## 4. Reasoning (Thinking Mode)

추론 과정을 단계별로 보여준 뒤 최종 답변을 생성합니다.

In [ ]:
in_answer = False

stream = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "닭과 토끼가 합쳐서 20마리, 다리 합계가 56개일 때 각각 몇 마리?"}],
    temperature=1.0,
    max_tokens=16384,
    extra_body={
        "reasoning_budget": 8192,
        "chat_template_kwargs": {"enable_thinking": True},
    },
    stream=True,
)

print("=== 추론 과정 ===")
for chunk in stream:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    reasoning = getattr(delta, "reasoning_content", None)
    content = delta.content

    if reasoning:
        print(reasoning, end="", flush=True)
    elif content:
        if not in_answer:
            print("\n\n=== 최종 답변 ===")
            in_answer = True
        print(content, end="", flush=True)
print()

## 5. Function Calling (Tool Use)

LLM이 외부 함수 호출 시점을 판단하고, 인자를 JSON으로 생성합니다.

In [ ]:
# 도구 정의
tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "특정 도시의 현재 날씨를 조회합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "도시 이름 (예: 서울, 부산, Tokyo)"}
            },
            "required": ["location"],
        },
    },
}]

# Mock 함수
def get_weather(location: str) -> str:
    mock = {"서울": "맑음, 18°C, 습도 55%", "부산": "흐림, 22°C, 습도 70%"}
    return mock.get(location, f"{location}: 정보 없음")

# 1차 요청
messages = [{"role": "user", "content": "서울 날씨가 어때?"}]
response = client.chat.completions.create(
    model=CHAT_MODEL, messages=messages, tools=tools, tool_choice="auto", max_tokens=256,
)
msg = response.choices[0].message

# tool_calls 처리 → 2차 요청
if msg.tool_calls:
    for tool_call in msg.tool_calls:
        fn_args = json.loads(tool_call.function.arguments)
        result = get_weather(**fn_args)
        print(f"[도구 호출] {tool_call.function.name}({fn_args}) → {result}")
        messages.append(msg)
        messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
    final = client.chat.completions.create(model=CHAT_MODEL, messages=messages, max_tokens=256)
    print(f"\n최종 답변: {final.choices[0].message.content}")
else:
    print(f"답변 (도구 미사용): {msg.content}")

## 6. 자유 실험

아래 셀에서 자유롭게 프롬프트를 바꿔가며 테스트해보세요.

In [ ]:
response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "user", "content": "여기에 원하는 프롬프트를 입력하세요"},
    ],
    temperature=0.7,
    max_tokens=512,
)

print(response.choices[0].message.content)